# **EEG Signal Quality Analysis**

This Notebook may be used for an OpenBCI GUI EEG recording structure in **60s blocks that alternate between Eyes Open (EO) condition and Eyes Closed (EC)**. Complete registration was accomplished by contributor in 11 minutes in total.

Through the obtained `.txt` generated file containing raw EEG data this Notebook displays:

1. **Valid Channels**: discards railed electrodes or the ones with bad conductivity.
2. **Relative Energy per frequency band (EC vs EO)**: Delta, Theta, Alpha, Beta and Gamma. **Alpha activity (8-13 Hz)** should increase with **EC** as a testing signal quality condition.
3. **Spectrogram from all channels and epochs**: each epoch corresponds to one minute. This time-frequency domain graph is shown in lineal scale and logarithmic. Each block/epoch is marked as well as the alpha activity band.


## ***Initial Configuration***

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import welch, spectrogram, butter, iirnotch, tf2sos, sosfiltfilt

DATA_FILE = "recording_11_min_EO_EC.txt"   # PASTE HERE THE NAME OF YOUR RECORDING SESSION FILE

fs         = 250                     # Hz Sampling frequency (OpenBCI Cyton)
BLOCK_SEC  = 60                      # s per block (epoch EO/EC)
EYES_OPEN_FIRST = True               # True: block 0 = EO, 1 = EC, 2 = EO...  
BLOCK_SAMPLES = fs * BLOCK_SEC
TRIM_SEC   = 5                       # s discarded at each block edge (reject EO/EC transients)
TRIM_SAMPLES = fs * TRIM_SEC

EXG_CHANNELS = ["Fp1", "Fp2", "C3", "C4", "P7", "P8", "O1", "O2"]   

BANDS = {"Delta": (0.5, 4), "Theta": (4, 8), "Alpha": (8, 13), "Beta": (13, 32), "Gamma": (32, 45)}
BAND_COLORS = {"Delta": "coral", "Theta": "green", "Alpha": "teal", "Beta": "brown", "Gamma": "navy"}

BP_LO, BP_HI = 0.5, 45               # Bandpass filter as the OpenBCI GUI does
NOTCH_FUND   = 50                    # Notch filter as the OpenBCI GUI does

# Welch parameters
NPERSEG  = fs * 5                    # 1250 samples to gain a resolution of 0.2 Hz
NOVERLAP = NPERSEG // 2

# Spectrogram parameters 
SPEC_NPERSEG = 512
SPEC_NOVERLAP = 256
SPEC_FMAX = 125                      

SAVE_FIGURES = True

print("Configuration loaded.")

## ***Helper Functions***

In [ ]:
# Depending on NumPy version it is compatible one or the other
_trapz = np.trapezoid if hasattr(np, "trapezoid") else np.trapz


def load_data(path):
    df = pd.read_csv(path, comment="%")
    df.columns = df.columns.str.strip()
    return df


def clean_segment(seg):
    seg = np.asarray(seg, float).copy()
    seg[seg == 0] = np.nan
    nans = np.isnan(seg)
    if nans.mean() > 0.30:
        return None
    idx = np.arange(len(seg))
    seg[nans] = np.interp(idx[nans], idx[~nans], seg[~nans])
    return seg


def build_bandpass(flo=BP_LO, fhi=BP_HI, order=4):
    nyq = fs / 2
    fhi = min(fhi, nyq - 1)
    return butter(order, [flo/nyq, fhi/nyq], btype="band", output="sos")


def build_notch(fund=NOTCH_FUND, Q=30):
    b, a = iirnotch(fund, Q=Q, fs=fs)
    return tf2sos(b, a)


SOS_BP    = build_bandpass()
SOS_NOTCH = build_notch()


def block_condition(b_idx):
    is_eo = (b_idx % 2 == 0) == EYES_OPEN_FIRST
    return "EO" if is_eo else "EC"

def block_segment(values, b_idx):
    s = b_idx * BLOCK_SAMPLES + TRIM_SAMPLES
    e = s + BLOCK_SAMPLES - 2 * TRIM_SAMPLES
    return clean_segment(values[s:e])


def psd_block(values, b_idx, sos):
    seg = block_segment(values, b_idx)
    if seg is None:
        return np.linspace(0, fs/2, NPERSEG//2 + 1), None
    seg = sosfiltfilt(sos, seg)
    f, Pxx = welch(seg, fs, nperseg=NPERSEG, noverlap=NOVERLAP, window="hann")
    pmax = Pxx.max()
    return f, (Pxx/pmax if pmax > 0 else Pxx)


def energy_band(psd, f, f1, f2):
    mask = (f >= f1) & (f <= f2)
    return float(_trapz(psd[mask], f[mask])) if mask.any() else 0.0


def detect_valid_channels(df, channels, n_blocks):
    power = {}
    for c in channels:
        psds = [psd_block(df[c].values, b, SOS_BP)[1] for b in range(n_blocks)]
        psds = [p for p in psds if p is not None]
        if psds:
            power[c] = float(np.mean(psds))
    if not power:
        return [], list(channels)
    threshold = np.median(list(power.values())) / 10
    valids   = [c for c, p in power.items() if p >  threshold]
    discarded = [c for c, p in power.items() if p <= threshold]
    return valids, discarded


print("Functions ready.")

## ***Load Cell***

In [ ]:
df = load_data(DATA_FILE)
print(f"{len(df)} samples ({len(df)/fs:.1f} s)")

channels = [c for c in df.columns if "EXG Channel" in c][:8]
if not channels:
    raise ValueError(f"Columns'EXG Channel' were not found. Columns: {list(df.columns)}")
POS = {i: (EXG_CHANNELS[i] if i < len(EXG_CHANNELS) else f"ch{i}") for i in range(len(channels))}
print("channels EXG detected:")
for i, c in enumerate(channels):
    print(f'  [{i}] "{c}"  ->  {POS[i]}')

n_blocks = int(np.floor(len(df) / BLOCK_SAMPLES))
print(f"\nBlocks of {BLOCK_SEC}s: {n_blocks}  ({', '.join(block_condition(b) for b in range(n_blocks))})")

channels_valid, channels_discarded = detect_valid_channels(df, channels, n_blocks)
label_channels = lambda c: POS[channels.index(c)]
print(f"\n channels valid:    {[label_channels(c) for c in channels_valid]}")
print(f"channels discarded:  {[label_channels(c) for c in channels_discarded]}")

def save_figure(fig, name):
    if SAVE_FIGURES:
        fig.savefig(name, dpi=200, bbox_inches="tight", facecolor="white")
        print(f"  saved as: {name}")

print("\nData ready")

# ***Relative Energy per Band (EC vs EO)***

In [ ]:
rel_energy = {c: {b: {"EC": [], "EO": []} for b in BANDS} for c in channels_valid}
for c in channels_valid:
    for b_idx in range(n_blocks):
        f, psd = psd_block(df[c].values, b_idx, SOS_BP)
        if psd is None:
            continue
        cond    = block_condition(b_idx)
        e_total = energy_band(psd, f, f.min(), f.max())
        for freq_band, (f1, f2) in BANDS.items():
            e = energy_band(psd, f, f1, f2) / e_total if e_total > 0 else 0
            rel_energy[c][freq_band][cond].append(e)


def bar_band(ax, freq_band, e_EC, e_EO, color, show_ylabel=False):
    def bar(x_label, values, alpha_fill, marker):
        if not values:
            return
        m, s = np.mean(values), np.std(values)
        ax.bar(x_label, m, yerr=s, color=color, edgecolor="black", alpha=alpha_fill,
               capsize=7, width=0.5, error_kw=dict(linewidth=1.3))
        ax.scatter([x_label]*len(values), values, color="black", zorder=5, s=38, marker=marker)
    bar("EC", e_EC, 1.0, "o")
    bar("EO", e_EO, 0.45, "D")
    f1, f2 = BANDS[freq_band]
    ax.set_title(f"{freq_band}\n({f1}–{f2} Hz)", fontsize=16, fontweight="bold")
    ax.tick_params(axis="both", labelsize=13)
    for tick in ax.get_xticklabels():
        tick.set_fontweight("bold")
    if show_ylabel:
        ax.set_ylabel("Relative energy", fontsize=16, fontweight="bold")
    ax.grid(axis="y", linestyle="--", alpha=0.4)


fig, axes = plt.subplots(1, len(BANDS), figsize=(3.4*len(BANDS), 5), sharey=True)
fig.suptitle("EC epochs (circles)   EO epochs (diamonds)\n"
             f"Relative energy per frequency band: EC vs EO  ({BP_LO}–{BP_HI} Hz filter)",
             fontsize=16, fontweight="bold")
for ax, (freq_band, color) in zip(axes, BAND_COLORS.items()):
    e_EC = [e for c in channels_valid for e in rel_energy[c][freq_band]["EC"]]
    e_EO = [e for c in channels_valid for e in rel_energy[c][freq_band]["EO"]]
    bar_band(ax, freq_band, e_EC, e_EO, color, show_ylabel=(ax is axes[0]))
plt.tight_layout()
save_figure(fig, "relative_energy_EC_vs_EO.png")
plt.show()


# ***Spectrogram***

In [ ]:
def plot_spectrogram(log_freq=False, fmax=SPEC_FMAX):
    sigs = [clean_segment(df[c].values) for c in channels_valid]
    sigs = [s for s in sigs if s is not None]
    if not sigs:
        print("Not any valid channel"); return
    sig_avg = np.mean(sigs, axis=0)

    for label, sos in [("Raw (no filter)", None), ("Notch filter", SOS_NOTCH)]:
        sig = sosfiltfilt(sos, sig_avg) if sos is not None else sig_avg
        f_s, t_s, Sxx = spectrogram(sig, fs=fs, nperseg=SPEC_NPERSEG,
                                    noverlap=SPEC_NOVERLAP, window="hann")
        Sxx_dB = 10*np.log10(Sxx + 1e-12)

        fig, ax = plt.subplots(figsize=(15, 5))
        im = ax.pcolormesh(t_s, f_s, Sxx_dB, shading="gouraud", cmap="RdYlBu_r",
                           vmin=np.percentile(Sxx_dB, 5), vmax=np.percentile(Sxx_dB, 99))
        cbar = fig.colorbar(im, ax=ax); cbar.set_label("Power (dB)", fontsize=14, fontweight="bold")

        if log_freq:
            ax.set_yscale("log"); ax.set_ylim(1, fmax)
        else:
            ax.set_ylim(0, fmax)

        for blk in range(n_blocks + 1):
            ax.axvline(blk*BLOCK_SEC, color="white", linewidth=2)
        for blk in range(n_blocks):
            ax.text(blk*BLOCK_SEC + BLOCK_SEC/2, fmax*0.93, block_condition(blk),
                    ha="center", va="top", fontsize=11, color="black", fontweight="bold")
        for freq in (8, 13):                      
            ax.axhline(freq, linestyle="--", linewidth=1.6, alpha=0.8, color="black")

        ax.set_xlim(0, n_blocks*BLOCK_SEC)
        ax.set_xlabel("Time (s)", fontsize=14, fontweight="bold")
        ax.set_ylabel("Frequency (Hz)", fontsize=14, fontweight="bold")
        eje = "log freq axis" if log_freq else "linear freq axis"
        ax.set_title(f"Spectrogram — average of valid channels ({label}) · {eje}",
                     fontsize=14, fontweight="bold")
        plt.tight_layout()
        tag = ("notch" if sos is not None else "raw") + ("_log" if log_freq else "_lin")
        save_figure(fig, f"spectrogram_{tag}.png")
        plt.show()


print(" Lineal Scale")
plot_spectrogram(log_freq=False)

print(" Logarithmic Scale ")
plot_spectrogram(log_freq=True)
